In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Configure pandas display
pd.set_option("display.max_columns", None)


In [2]:
data_path = "../suno_ai_posts_with_genres.jsonl"
df = pd.read_json(data_path, lines=True)

In [3]:
# Rename and transform columns to make the data more understandable
filtered_df_with_suno_downloaded_songs = df[
    df["download_path"].str.contains("suno", na=False)
].copy()

# Select and rename columns
filtered_df_with_suno_downloaded_songs = filtered_df_with_suno_downloaded_songs[
    [
        "id",
        "author",
        "url",
        "title",
        "download_path",
        "link_flair_text",
        "url_overridden_by_dest",
        "permalink",
        "selftext",
        "predicted_genres",
        "top_3_genres",
        "genre_scores",
        "over_18",
        "retrieved_on",
        "num_comments",
        "ups",
        "downs",
    ]
]

# Rename columns for better clarity
filtered_df_with_suno_downloaded_songs = filtered_df_with_suno_downloaded_songs.rename(
    columns={
        "link_flair_text": "flair",
        "url_overridden_by_dest": "suno_url",
        "permalink": "reddit_path",
        "selftext": "post_text",
    }
)

# Add full Reddit URL by concatenating
filtered_df_with_suno_downloaded_songs["reddit_url"] = (
    "https://www.reddit.com" + filtered_df_with_suno_downloaded_songs["reddit_path"]
)

# Drop the now-unnecessary reddit_path column
filtered_df_with_suno_downloaded_songs = filtered_df_with_suno_downloaded_songs.drop(
    columns=["reddit_path"]
)

# Display
filtered_df_with_suno_downloaded_songs.head()

,id,author,url,title,download_path,flair,suno_url,post_text,predicted_genres,top_3_genres,genre_scores,over_18,retrieved_on,num_comments,ups,downs,reddit_url
5122,1c6q7op,GewoonRobinn13,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,Made a song about being a plant parent. I hope...,dataset/suno/1c6q7op.mp3,Song,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,,[],"[jazz, country, folk]","[0.2537326515, 0.1759615242, 0.16200187800000002]",False,1713401009,0,1,0,https://www.reddit.com/r/SunoAI/comments/1c6q7...
5125,1c6qcaa,GewoonRobinn13,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,I made a song about being a plant parent. I ho...,dataset/suno/1c6qcaa.mp3,Song,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,,[],"[jazz, country, folk]","[0.2537326515, 0.1759615242, 0.16200187800000002]",False,1713401377,0,0,0,https://www.reddit.com/r/SunoAI/comments/1c6qc...
5127,1c6qesm,Cardboard_Chef,https://suno.com/song/c7fd4cbd-c432-4ab5-831b-...,I think Suno does and incredible job at making...,dataset/suno/1c6qesm.mp3,Song,https://suno.com/song/c7fd4cbd-c432-4ab5-831b-...,,[],"[electronic, pop, rock]","[0.3871032596, 0.2404211164, 0.2153102458]",False,1713401576,4,5,0,https://www.reddit.com/r/SunoAI/comments/1c6qe...
5129,1c6qfbw,RDCK78,https://www.reddit.com/r/SunoAI/comments/1c6qf...,"First try at a country song, first try at any ...",dataset/suno/1c6qfbw.mp3,Song,None,https://suno.com/song/16c93478-bb0f-4ea4-99ba-...,[pop],"[pop, rock, poprock]","[0.6949518919000001, 0.4699110687, 0.1709648073]",False,1713401617,6,2,0,https://www.reddit.com/r/SunoAI/comments/1c6qf...
5130,1c6qhdw,FluffyBrewbs,https://suno.com/song/2326aac2-42b4-4cb7-a428-...,Didn't think AI would make me cry,dataset/suno/1c6qhdw.mp3,Song,https://suno.com/song/2326aac2-42b4-4cb7-a428-...,I wrote lyrics to a song a couple years ago an...,[classical],"[classical, soundtrack, easylistening]","[0.8194063902000001, 0.38395628330000003, 0.24...",False,1713401782,4,0,0,https://www.reddit.com/r/SunoAI/comments/1c6qh...


In [4]:
import re

def extract_suno_url(selftext: str) -> str | None:
    """Function to extract Suno URLs from text"""
    selftext = selftext.replace(r"\/", "/")
    match = re.search(r"https://suno\.com/[^\s\)\]\}]+", selftext)
    if match:
        return match.group(0)
    return None


In [11]:
# Create a copy of the dataframe to avoid modifying the original
processed_df = filtered_df_with_suno_downloaded_songs.copy()

# For rows where suno_url is missing but post_text contains a Suno URL
mask = processed_df["suno_url"].isna()
processed_df.loc[mask, "extracted_url"] = processed_df.loc[mask, "post_text"].apply(
    extract_suno_url
)

# Update suno_url with extracted URLs
processed_df.loc[mask & processed_df["extracted_url"].notna(), "suno_url"] = (
    processed_df.loc[mask & processed_df["extracted_url"].notna(), "extracted_url"]
)

# Drop the temporary column
processed_df = processed_df.drop("extracted_url", axis=1)

# Count how many URLs were found
urls_found = (
    processed_df["suno_url"].notna().sum()
    - filtered_df_with_suno_downloaded_songs["suno_url"].notna().sum()
)
print(f"Found and added {urls_found} Suno URLs from post_text")

# Display the updated DataFrame
processed_df.head()

Found and added 1381 Suno URLs from post_text


,id,author,url,title,download_path,flair,suno_url,post_text,predicted_genres,top_3_genres,genre_scores,over_18,retrieved_on,num_comments,ups,downs,reddit_url
5122,1c6q7op,GewoonRobinn13,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,Made a song about being a plant parent. I hope...,dataset/suno/1c6q7op.mp3,Song,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,,[],"[jazz, country, folk]","[0.2537326515, 0.1759615242, 0.16200187800000002]",False,1713401009,0,1,0,https://www.reddit.com/r/SunoAI/comments/1c6q7...
5125,1c6qcaa,GewoonRobinn13,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,I made a song about being a plant parent. I ho...,dataset/suno/1c6qcaa.mp3,Song,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,,[],"[jazz, country, folk]","[0.2537326515, 0.1759615242, 0.16200187800000002]",False,1713401377,0,0,0,https://www.reddit.com/r/SunoAI/comments/1c6qc...
5127,1c6qesm,Cardboard_Chef,https://suno.com/song/c7fd4cbd-c432-4ab5-831b-...,I think Suno does and incredible job at making...,dataset/suno/1c6qesm.mp3,Song,https://suno.com/song/c7fd4cbd-c432-4ab5-831b-...,,[],"[electronic, pop, rock]","[0.3871032596, 0.2404211164, 0.2153102458]",False,1713401576,4,5,0,https://www.reddit.com/r/SunoAI/comments/1c6qe...
5129,1c6qfbw,RDCK78,https://www.reddit.com/r/SunoAI/comments/1c6qf...,"First try at a country song, first try at any ...",dataset/suno/1c6qfbw.mp3,Song,https://suno.com/song/16c93478-bb0f-4ea4-99ba-...,https://suno.com/song/16c93478-bb0f-4ea4-99ba-...,[pop],"[pop, rock, poprock]","[0.6949518919000001, 0.4699110687, 0.1709648073]",False,1713401617,6,2,0,https://www.reddit.com/r/SunoAI/comments/1c6qf...
5130,1c6qhdw,FluffyBrewbs,https://suno.com/song/2326aac2-42b4-4cb7-a428-...,Didn't think AI would make me cry,dataset/suno/1c6qhdw.mp3,Song,https://suno.com/song/2326aac2-42b4-4cb7-a428-...,I wrote lyrics to a song a couple years ago an...,[classical],"[classical, soundtrack, easylistening]","[0.8194063902000001, 0.38395628330000003, 0.24...",False,1713401782,4,0,0,https://www.reddit.com/r/SunoAI/comments/1c6qh...


In [6]:
# Filter out entries with "Meme Song" flair
filtered_processed_df = processed_df[processed_df["flair"] != "Meme Song"]

# Display information about the filtering
meme_song_count = len(processed_df) - len(filtered_processed_df)
print(f"Removed {meme_song_count} entries with 'Meme Song' flair")
print(f"Remaining entries: {len(filtered_processed_df)}")

# Show the first few rows of the filtered dataframe
filtered_processed_df.head()

Removed 411 entries with 'Meme Song' flair
Remaining entries: 4059


,id,author,url,title,download_path,flair,suno_url,post_text,predicted_genres,top_3_genres,genre_scores,over_18,retrieved_on,num_comments,ups,downs,reddit_url
5122,1c6q7op,GewoonRobinn13,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,Made a song about being a plant parent. I hope...,dataset/suno/1c6q7op.mp3,Song,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,,[],"[jazz, country, folk]","[0.2537326515, 0.1759615242, 0.16200187800000002]",False,1713401009,0,1,0,https://www.reddit.com/r/SunoAI/comments/1c6q7...
5125,1c6qcaa,GewoonRobinn13,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,I made a song about being a plant parent. I ho...,dataset/suno/1c6qcaa.mp3,Song,https://suno.com/song/b99e8ada-0a7f-496b-ab32-...,,[],"[jazz, country, folk]","[0.2537326515, 0.1759615242, 0.16200187800000002]",False,1713401377,0,0,0,https://www.reddit.com/r/SunoAI/comments/1c6qc...
5127,1c6qesm,Cardboard_Chef,https://suno.com/song/c7fd4cbd-c432-4ab5-831b-...,I think Suno does and incredible job at making...,dataset/suno/1c6qesm.mp3,Song,https://suno.com/song/c7fd4cbd-c432-4ab5-831b-...,,[],"[electronic, pop, rock]","[0.3871032596, 0.2404211164, 0.2153102458]",False,1713401576,4,5,0,https://www.reddit.com/r/SunoAI/comments/1c6qe...
5129,1c6qfbw,RDCK78,https://www.reddit.com/r/SunoAI/comments/1c6qf...,"First try at a country song, first try at any ...",dataset/suno/1c6qfbw.mp3,Song,https://suno.com/song/16c93478-bb0f-4ea4-99ba-...,https://suno.com/song/16c93478-bb0f-4ea4-99ba-...,[pop],"[pop, rock, poprock]","[0.6949518919000001, 0.4699110687, 0.1709648073]",False,1713401617,6,2,0,https://www.reddit.com/r/SunoAI/comments/1c6qf...
5130,1c6qhdw,FluffyBrewbs,https://suno.com/song/2326aac2-42b4-4cb7-a428-...,Didn't think AI would make me cry,dataset/suno/1c6qhdw.mp3,Song,https://suno.com/song/2326aac2-42b4-4cb7-a428-...,I wrote lyrics to a song a couple years ago an...,[classical],"[classical, soundtrack, easylistening]","[0.8194063902000001, 0.38395628330000003, 0.24...",False,1713401782,4,0,0,https://www.reddit.com/r/SunoAI/comments/1c6qh...


In [7]:
# Export the processed dataset to CSV
filtered_processed_df.to_csv('suno_ai_processed_dataset.csv', index=False)

# Confirmation message
print(f"Dataset successfully exported to 'suno_ai_processed_dataset.csv'")
print(f"Total records: {len(filtered_processed_df)}")

Dataset successfully exported to 'suno_ai_processed_dataset.csv'
Total records: 4059
